In [48]:
import numpy as np
import cv2
from tqdm import tqdm
from glob import glob
from pathlib import Path
import json
!pip install kaggle --upgrade
import os
import zipfile

In [35]:
train_data = '/kaggle/input/fruits/fruits-360_100x100/fruits-360/Training'
test_data = '/kaggle/input/fruits/fruits-360_100x100/fruits-360/Test'
output_path_train = '/kaggle/working/masks_data_100x100/train'
output_path_test = '/kaggle/working/masks_data_100x100/test'

In [36]:
fruit_folders = sorted(glob(train_data + '/*'))
print(len(fruit_folders))
fruit_folders = sorted(glob(test_data + '/*'))
print(len(fruit_folders))

221
221


In [38]:
def generate_masks(data_path, output_path):
    fruit_folders = sorted(glob(data_path + '/*'))
    # input image folders
    for fruit_folder in fruit_folders:
        fruit_name = fruit_folder.split('/')[-1]
        image_paths = glob(fruit_folder + '/*.jpg')

        # output folders
        image_out_folder = Path(f'{output_path}/{fruit_name}/Images')
        mask_out_folder = Path(f'{output_path}/{fruit_name}/Masks')

        # create directories (like mkdir -p)
        image_out_folder.mkdir(parents=True, exist_ok=True)
        mask_out_folder.mkdir(parents=True, exist_ok=True)

        # process all images
        for image_path in tqdm(image_paths, desc=f'Generating masks for {fruit_name}', ncols=100):
            image = cv2.imread(image_path)
            if image is None:
                continue

            # converting images to grayscale
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # create binary mask, fruit → 255, background → 0
            _, mask = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

            # morphological cleanup
            kernel = np.ones((3, 3), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            # saves both image and mask
            base = image_path.split('/')[-1].split('.')[0]
            cv2.imwrite(str(image_out_folder / f'{base}.jpg'), image)
            cv2.imwrite(str(mask_out_folder / f'{base}_mask.jpg'), mask)


In [19]:
generate_masks(train_data, output_path_train)

Generating masks for Zucchini dark 1: 100%|██████████████████████| 240/240 [00:02<00:00, 111.97it/s]


In [24]:
generate_masks(test_data, output_path_test)

Generating masks for Zucchini dark 1: 100%|████████████████████████| 80/80 [00:00<00:00, 120.51it/s]


In [49]:
!mkdir -p ~/.kaggle
!cp /kaggle/input/kaggle-api-key/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!ls -R /kaggle/working/masks_data_100x100 | head -n 40

/kaggle/working/masks_data_100x100:
test
train

/kaggle/working/masks_data_100x100/test:
Apple 10
Apple 11
Apple 12
Apple 13
Apple 14
Apple 17
Apple 18
Apple 19
Apple 5
Apple 6
Apple 7
Apple 8
Apple 9
Apple Braeburn 1
Apple Core 1
Apple Crimson Snow 1
Apple Golden 1
Apple Golden 2
Apple Golden 3
Apple Granny Smith 1
Apple hit 1
Apple Pink Lady 1
Apple Red 1
Apple Red 2
Apple Red 3
Apple Red Delicious 1
Apple Red Yellow 1
Apple Red Yellow 2
Apple Rotten 1
Apple worm 1
Apricot 1
Avocado 1
Avocado Black 1
Avocado Green 1
Avocado ripe 1
ls: write error: Broken pipe


In [40]:
def zip_with_progress(folder_path, output_path):
    # total files
    file_paths = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_paths.append(os.path.join(root, file))

    # zip with progress bar
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in tqdm(file_paths, desc="Zipping dataset", unit="file"):
            arcname = os.path.relpath(file, folder_path)
            zipf.write(file, arcname)

# zip data
zip_with_progress('/kaggle/working/masks_data_100x100', '/kaggle/working/fruit360_segmentation_data_100x100.zip')

Zipping dataset: 100%|██████████| 305330/305330 [02:43<00:00, 1870.66file/s]


In [50]:
ls -lh /kaggle/working/*.zip

-rw-r--r-- 1 root root 1.1G Oct  8 06:56 /kaggle/working/fruit360_segmentation_data_100x100.zip


In [43]:
metadata = {
    "title": "Fruit360 Segmentation Masks 100x100",
    "id": "sehrishnoreen31/fruit360-segmentation-masks-100x100",
    "licenses": [{"name": "CC BY 4.0"}],
    "description": (
        "This dataset contains 100x100 fruit images from the Fruit-360 dataset "
        "and their automatically generated segmentation masks. "
        "Each fruit class has 'Images' and 'Masks' folders for both 'train' and 'test' sets. "
        "Masks were generated using OpenCV thresholding techniques optimized for transparent or white backgrounds."
    )
}
with open("/kaggle/working/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [59]:
!mkdir -p /kaggle/working/upload_data

In [60]:
# moving data to a single folder --
!mv /kaggle/working/fruit360_segmentation_data_100x100.zip /kaggle/working/upload_data/
!mv /kaggle/working/dataset-metadata.json /kaggle/working/upload_data/

In [62]:
!ls -lh /kaggle/working/upload_data

total 1.1G
-rw-r--r-- 1 root root  532 Oct  8 06:58 dataset-metadata.json
-rw-r--r-- 1 root root 1.1G Oct  8 06:56 fruit360_segmentation_data_100x100.zip


In [63]:
# upload data
!kaggle datasets create -p /kaggle/working/upload_data -r zip --dir-mode zip

Starting upload for file fruit360_segmentation_data_100x100.zip
100%|██████████████████████████████████████| 1.00G/1.00G [00:41<00:00, 26.1MB/s]
Upload successful: fruit360_segmentation_data_100x100.zip (1GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/sehrishnoreen31/fruit360-segmentation-masks-100x100
